In [5]:
# ============================================================
# DIRECT / WHOLE-RECORD SUMMARIZATION
#
# Final study workflow:
# complete deduplicated patient record -> Luna -> final summary
# ============================================================

import json
import sys
import time
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("../..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

In [6]:
from src.llm.llm import generate_summary
from config.prompts import SYSTEM_PROMPT




In [7]:
# ============================================================
# LOAD + PREPROCESS SOURCE NOTES
# ============================================================

notes = pd.read_csv(
    PROJECT_ROOT
    / "data"
    / "raw"
    / "clinical_notes.csv"
)

notes_clean = notes[
    notes["clean_note_text"]
    .astype(str)
    .str.strip()
    != "#NAME?"
].copy()

notes_dedup = (
    notes_clean
    .sort_values(
        ["person_id", "creation_timestamp"]
    )
    .drop_duplicates(
        subset=[
            "person_id",
            "clean_note_text",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)

print("Raw notes:", len(notes))
print("After cleaning + deduplication:", len(notes_dedup))
print("Patients:", notes_dedup["person_id"].nunique())

Raw notes: 1602
After cleaning + deduplication: 1103
Patients: 50


In [12]:
# ============================================================
# LOAD FROZEN 20-PATIENT STUDY COHORT
# ============================================================

COHORT_PATH = (
    PROJECT_ROOT
    / "data"
    / "patients"
    / "study_patient_ids.json"
)

with COHORT_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    STUDY_PATIENT_IDS = json.load(file)

print("Study patients:", len(STUDY_PATIENT_IDS))

assert len(STUDY_PATIENT_IDS) == 50

Study patients: 50


In [9]:
# ============================================================
# DIRECT WORKFLOW OUTPUT
# ============================================================

DIRECT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "direct"
)

DIRECT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUT_PATH = (
    DIRECT_DIR
    / "direct_summaries.json"
)

if OUTPUT_PATH.exists():
    with OUTPUT_PATH.open(
        "r",
        encoding="utf-8",
    ) as file:
        direct_results = json.load(file)
else:
    direct_results = {}

print("Already completed:", len(direct_results))

Already completed: 0


In [13]:
# ============================================================
# GENERATE DIRECT SUMMARIES
#
# Each patient receives the complete cleaned/deduplicated
# chronological record.
# ============================================================

total_patients = len(STUDY_PATIENT_IDS)


for i, patient_id in enumerate(
    STUDY_PATIENT_IDS,
    start=1,
):

    if patient_id in direct_results:
        print(
            f"[{i}/{total_patients}] Skipping completed: "
            f"{patient_id}"
        )
        continue

    patient_notes = (
        notes_dedup[
            notes_dedup["person_id"]
            == patient_id
        ]
        .sort_values("creation_timestamp")
        .reset_index(drop=True)
    )

    print(
        f"\n[{i}/{total_patients}] Patient: {patient_id} | "
        f"{len(patient_notes)} notes"
    )

    start = time.time()

    summary = generate_summary(
        patient_id,
        patient_notes,
        SYSTEM_PROMPT,
    )

    latency = time.time() - start

    direct_results[patient_id] = {
        "person_id": patient_id,
        "source_note_count": len(patient_notes),
        "summary": summary,
        "latency_seconds": round(
            latency,
            2,
        ),
    }

    # Save after every patient.
    with OUTPUT_PATH.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            direct_results,
            file,
            indent=2,
            ensure_ascii=False,
        )

    print(
        f"Saved: {patient_id} "
        f"({latency:.1f}s)"
    )

    time.sleep(2)


print(
    "\nCompleted direct summaries:",
    len(direct_results),
)

[1/50] Skipping completed: 028998ee-babc-4096-9b28-001bc2f9a84e
[2/50] Skipping completed: 04df53ea-55c1-48d9-84a1-1f15c133b29b
[3/50] Skipping completed: 05192757-942f-460d-b4ff-004ec39cc5ee
[4/50] Skipping completed: 0c04d9fb-d8d0-4ee0-8fc9-0c3f7f148bdf
[5/50] Skipping completed: 0f438665-d430-4adb-8acc-c3beed9e4942
[6/50] Skipping completed: 136c7916-4f9b-4e5c-bf01-77e9d2c681a2
[7/50] Skipping completed: 137b8481-4f1d-4b7f-babd-20f7117023ad
[8/50] Skipping completed: 1705dd0f-011a-492c-b006-b27e03f2f4ed
[9/50] Skipping completed: 1dbe23dc-0d1e-431b-81eb-497282b46a14
[10/50] Skipping completed: 28570119-9cdc-4120-98c0-4edb76cf36a3
[11/50] Skipping completed: 29ea304f-821d-474e-81a1-394ca3945e02
[12/50] Skipping completed: 31f9612b-5a6b-48ea-887b-895772a83b99
[13/50] Skipping completed: 359014a1-10e6-4bd8-9ba7-513d021c971e
[14/50] Skipping completed: 37b5ce4d-dcfd-4bb7-bee4-d597eb114703
[15/50] Skipping completed: 420df33b-0072-4124-b1c6-0589daef3677
[16/50] Skipping completed: 42149a